In [ ]:
import json
import pandas as pd
import numpy as np
import os
import torch
import timesfm



def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
with open(path, "r") as f:
    dataset_days = json.load(f)

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

PREDICTION_LENGTH = 96
MAX_CONTEXT = 2048

torch.set_float32_matmul_precision("high")

model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch"
)

model.compile(
    timesfm.ForecastConfig(
        max_context=MAX_CONTEXT,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

rmse_results = []

for country in countries:
    print("Processing country:", country)

    data_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [col for col in df.columns if col not in features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:
            s_train = df.loc[df.index < cutoff, household].dropna().tail(MAX_CONTEXT)
            s_future = df.loc[df.index >= cutoff, household].head(PREDICTION_LENGTH)

            if len(s_train) < 10 or len(s_future) < PREDICTION_LENGTH:
                print(f"      Skipping {household}: insufficient data")
                continue

            point_forecast, quantile_forecast = model.forecast(
                horizon=PREDICTION_LENGTH,
                inputs=[s_train.to_numpy(dtype=float)]
            )

            y_pred = np.asarray(point_forecast)[0, :PREDICTION_LENGTH]
            y_true = s_future.to_numpy(dtype=float)

            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=s_future.index)

            predictions_df_all_households[household] = y_pred

            rmse = root_mean_squared_error(y_true, y_pred)
            rmse_households.append(rmse)

        if len(rmse_households) == 0:
            print(f"      No valid households for {country} {day}")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        output = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

rmse_df = pd.DataFrame(rmse_results)

print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

Downloaded.
Processing country: Germany
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_day1_Germany.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_day2_Germany.csv
   Day: day3
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_day3_Germany.csv
   Day: day4
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_day4_Germany.csv
   Day: day5
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_day5_Germany.csv
Processing country: Ireland
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TimesFM2p5_pred_day1_Ireland.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predi